# STEP 1: Honda Data Loading Playground

**Purpose:** Visually explore and test Honda data loading process

**What This Notebook Does:**
1. Loads Honda Excel files
2. Shows sheet structure
3. Displays data at each step
4. Tests metadata extraction
5. Visualizes the wide-to-long transformation needed for Step 2

Run each cell to see the data at that stage.

## Setup: Import Libraries

# STEP 2: Data Extraction & Flattening Plan

## Goal
Convert wide-format data (trims as columns) into a flat long-format file with columns:
- `Description`, `Part Number`, `Cost`, `Residue`, `FRT`, `Price`, `Comments` (product data)
- `trim_level` (which trim), `applicability` (• = yes, E = exception, blank = no)

## Components of Data Extraction

### Component 1: Identify Section Boundaries
**What:** Find where each section (Packages, Electronics, Interior, Exterior, Cargo, General) starts and ends
**How:**
- Use section marker from metadata (e.g., "1.0 Packages and Kits" row)
- Extract rows between markers
- Handle sections with different markers (EN vs FR)

**Output:** Dictionary of section boundaries
```
{
  "Packages and Kits": {"start": 0, "end": 42},
  "Electronics": {"start": 43, "end": 78},
  ...
}
```

### Component 2: Identify Product Columns vs Trim Columns
**What:** Separate static product info from applicability columns
**How:**
- Product columns: Description, Part Number, Cost, Residue, FRT, Price, Comments
- Trim columns: All other columns (LX, SPORT, Si, Sport Touring, etc.)
- Handle newlines in column names (Excel splits on \n)

**Output:** Two lists
```
product_cols = ['Description', 'Part Number', 'Cost', 'FRT', 'Price', ...]
trim_cols = ['LX', 'SPORT', 'SPORT\nHybrid', 'Sport\nTouring', 'Si']
```

### Component 3: Clean Trim Column Names
**What:** Remove newlines and standardize trim names
**How:**
- Replace `\n` with space or hyphen
- Examples: `SPORT\nHybrid` → `SPORT Hybrid`, `Sport\nTouring` → `Sport Touring`
- Keep original text (don't lose info)

**Output:** Cleaned trim names
```
['LX', 'SPORT', 'SPORT Hybrid', 'Sport Touring', 'Si']
```

### Component 4: Extract Applicability Markers
**What:** Identify which trims each part applies to
**How:**
- Read cell values in trim columns
- Markers: `•` (yes), `E` (exception), `T` (test?), empty/NaN (no)
- Filter to only rows where at least one trim has applicability

**Output:** For each part × trim combination:
```
part_id=1, trim='LX', applicability='•'  → included
part_id=1, trim='Si', applicability=NaN  → excluded
```

### Component 5: Melt Wide to Long Format
**What:** Convert from one-row-per-part to one-row-per-part-trim
**How:**
- Use pd.melt() to reshape
- Keep product columns as id_vars
- Melt trim columns to rows
- Each row = part + trim + applicability

**Before:**
```
Description   Part#  LX  SPORT  Si
Wheel         123    •    •     •
Spoiler       456    •    NaN   NaN
```

**After:**
```
Description  Part#  trim_level  applicability
Wheel        123    LX          •
Wheel        123    SPORT       •
Wheel        123    Si          •
Spoiler      456    LX          •
Spoiler      456    SPORT       NaN
Spoiler      456    Si          NaN
```

### Component 6: Filter Applicability
**What:** Keep only rows where applicability is not NaN/empty
**How:**
- dropna(subset=['applicability'])
- Optionally keep exceptions (E) for review

**Output:** Flattened data with only applicable parts

### Component 7: Validate Column Mapping
**What:** Ensure all required columns exist and map correctly
**How:**
- Check if file has all expected product columns
- Handle missing columns (Cost, Residue, etc. may not exist)
- Map actual column names to standard output names

**Example:**
```
Expected: ['Description', 'Part Number', 'Cost', 'FRT', 'Price', 'Comments']
Actual:   ['Description', 'Part Number', 'Cost', 'FRT', 'Price', 'Comments']  ✓
OR
Actual:   ['Description', 'Part#', 'Cost', 'FRT', 'Price', 'Notes']  → map Part# → Part Number, Notes → Comments
```

## Process Flow

```
1. Load data sheet (civic 5d EN)
   ↓
2. Extract metadata (section marker, publication date) ✓ DONE
   ↓
3. Identify section boundaries
   ↓
4. For EACH SECTION:
   a. Identify product vs trim columns
   b. Clean trim column names
   c. Extract applicability markers
   d. Melt to long format
   d. Filter to applicable rows
   ↓
5. Combine all sections
   ↓
6. Validate column mapping
   ↓
7. Output: Flat CSV with all parts × trims
```

## Expected Output Columns

| Column | Source | Example | Notes |
|--------|--------|---------|-------|
| `Description` | Wide format row | "16\" Wheel" | Product description |
| `Part Number` | Wide format row | "25CIV-ENK-123" | Part identifier |
| `Cost` | Wide format row | "$1,234.56" | Cost value |
| `Residue` | Wide format row | (optional) | Residue value |
| `FRT` | Wide format row | "1.2" | Freight cost multiplier |
| `Price` | Wide format row | "$2,000.00" | Retail price |
| `Comments` | Wide format row | "Priced as a set..." | Notes |
| `trim_level` | Column name after melt | "LX", "SPORT", "Si" | Which trim |
| `applicability` | Cell value after melt | "•", "E", NaN | Yes/Exception/No |

## Implementation Order

1. **Step 2.1:** Identify section boundaries (rows 1-42, 43-78, etc.)
2. **Step 2.2:** Extract ONE section (e.g., Packages) to test
3. **Step 2.3:** Melt that section to long format
4. **Step 2.4:** Validate output (correct columns, correct trim names, correct applicability)
5. **Step 2.5:** Repeat for all sections
6. **Step 2.6:** Combine all sections into one flat file
7. **Step 2.7:** Final validation and quality checks

# STEP 2: DATA EXTRACTION & FLATTENING

## Step 2.1: Identify Section Boundaries

In [87]:
# Step 2.1: Identify Section Boundaries (AUTO-DETECT + COUNT EMPTY ROWS)

import re

def is_row_empty(row):
    """Check if a row is completely empty (all NaN or all whitespace)."""
    for val in row.values:
        if pd.notna(val):
            val_str = str(val).strip()
            if val_str and val_str != 'nan':
                return False
    return True

def identify_section_boundaries(df):
    """
    Find section boundaries in the data.
    Auto-detects sections by looking for section markers (e.g., "1.0", "1.1", "2.0", etc.)
    Returns dictionary with marker_row, data_start, data_end, total rows, empty rows, data rows.
    Does NOT include the marker row in counts.
    """
    
    df_str = df.astype(str)
    section_starts = []
    
    # Find all section markers (pattern: digit(s).digit(s) followed by text)
    section_pattern = r'^\d+\.\d+\s+(.+)$'
    
    for idx, row in df_str.iterrows():
        for cell_value in row.values:
            cell_str = str(cell_value).strip()
            match = re.match(section_pattern, cell_str)
            if match:
                section_name = match.group(1).strip()
                section_starts.append((idx, section_name, cell_str))
                break
    
    # Build section boundaries
    sections = {}
    for i, (marker_row, section_name, marker_text) in enumerate(section_starts):
        # Data starts AFTER the marker row
        data_start = marker_row + 1
        
        # Data ends BEFORE the next section marker (or at end of dataframe)
        if i + 1 < len(section_starts):
            next_marker_row = section_starts[i + 1][0]
            data_end = next_marker_row - 1
        else:
            # Last section: goes to end of dataframe
            data_end = len(df) - 1
        
        # Count rows in this section
        if data_start <= data_end:
            total_rows = data_end - data_start + 1
            
            # Count empty vs non-empty rows
            empty_rows = 0
            non_empty_rows = 0
            for row_idx in range(data_start, data_end + 1):
                if is_row_empty(df.iloc[row_idx]):
                    empty_rows += 1
                else:
                    non_empty_rows += 1
        else:
            total_rows = 0
            empty_rows = 0
            non_empty_rows = 0
        
        sections[section_name] = {
            "marker_row": marker_row,
            "data_start": data_start,
            "data_end": data_end,
            "total_rows": total_rows,
            "empty_rows": empty_rows,
            "non_empty_rows": non_empty_rows
        }
    
    return sections

# VERIFY: Use df_civic (from civic 5d.xlsx)
if 'df_civic' not in locals() or df_civic is None:
    print("❌ ERROR: df_civic not loaded!")
    print("   Make sure to run 'Load civic 5d.xlsx for Testing' cell first")
else:
    print(f"✓ Using file: civic 5d.xlsx")
    print(f"✓ Sheet: {en_sheet if 'en_sheet' in locals() else 'EN sheet'}")
    print(f"✓ Rows: {len(df_civic)}, Columns: {len(df_civic.columns)}\n")
    
    print("="*80)
    print("STEP 2.1: IDENTIFY SECTION BOUNDARIES (WITH EMPTY ROW COUNT)")
    print("="*80)
    
    sections = identify_section_boundaries(df_civic)
    
    if sections:
        print(f"\n✓ Found {len(sections)} sections:\n")
        total_all_rows = 0
        total_empty_rows = 0
        total_non_empty_rows = 0
        
        for section_name, bounds in sections.items():
            print(f"  {section_name}")
            print(f"    Marker row: {bounds['marker_row']}")
            print(f"    Rows: {bounds['data_start']} to {bounds['data_end']}")
            print(f"    Total rows in section: {bounds['total_rows']}")
            print(f"      - Empty rows: {bounds['empty_rows']}")
            print(f"      - Data rows: {bounds['non_empty_rows']}")
            print()
            
            total_all_rows += bounds['total_rows']
            total_empty_rows += bounds['empty_rows']
            total_non_empty_rows += bounds['non_empty_rows']
        
        print("="*80)
        print(f"SECTION TOTALS:")
        print(f"  Total rows (all sections): {total_all_rows}")
        print(f"  Empty rows: {total_empty_rows}")
        print(f"  Data rows: {total_non_empty_rows}")
        print(f"\nFILE TOTALS:")
        print(f"  Total rows in file: {len(df_civic)}")
        print(f"  Header/Marker/Footer rows: {len(df_civic) - total_all_rows}")
    else:
        print("\n❌ No sections found")

✓ Using file: civic 5d.xlsx
✓ Sheet: 26MY_CIV5D_APP_EN
✓ Rows: 147, Columns: 17

STEP 2.1: IDENTIFY SECTION BOUNDARIES (WITH EMPTY ROW COUNT)

✓ Found 6 sections:

  Packages and Kits
    Marker row: 9
    Rows: 10 to 47
    Total rows in section: 38
      - Empty rows: 7
      - Data rows: 31

  Wheels and Wheel Accessories
    Marker row: 48
    Rows: 49 to 60
    Total rows in section: 12
      - Empty rows: 1
      - Data rows: 11

  Exterior
    Marker row: 61
    Rows: 62 to 108
    Total rows in section: 47
      - Empty rows: 1
      - Data rows: 46

  Interior
    Marker row: 109
    Rows: 110 to 122
    Total rows in section: 13
      - Empty rows: 1
      - Data rows: 12

  Electrical
    Marker row: 123
    Rows: 124 to 130
    Total rows in section: 7
      - Empty rows: 1
      - Data rows: 6

  Paint Pens
    Marker row: 131
    Rows: 132 to 146
    Total rows in section: 15
      - Empty rows: 2
      - Data rows: 13

SECTION TOTALS:
  Total rows (all sections): 132
  E

## Step 2.2: Display Section Data

In [89]:
# Step 2.2: Display Section Data (WITH INDEX SUPPORT)

def display_section(section_ref, sections, df):
    """
    Display all rows for a given section with debugging info.
    
    Args:
        section_ref: Either a section name (string) or section index (integer, 0-based)
        sections: Dictionary of sections from identify_section_boundaries()
        df: DataFrame containing the data
    
    Examples:
        display_section("Packages and Kits", sections, df_civic)  # By name
        display_section(0, sections, df_civic)                    # By index (first section)
        display_section(2, sections, df_civic)                    # By index (third section)
    """
    
    # Convert section reference to section name
    section_names = list(sections.keys())
    
    if isinstance(section_ref, int):
        # Handle index-based access
        if section_ref < 0 or section_ref >= len(section_names):
            print(f"❌ Section index {section_ref} out of range (0-{len(section_names)-1})")
            print(f"\nAvailable sections:")
            for i, name in enumerate(section_names):
                bounds = sections[name]
                print(f"  [{i}] {name} (rows {bounds['data_start']}-{bounds['data_end']}, {bounds['non_empty_rows']} data rows)")
            return
        section_name = section_names[section_ref]
    else:
        # Handle name-based access
        section_name = section_ref
        if section_name not in sections:
            print(f"❌ Section '{section_name}' not found")
            print(f"\nAvailable sections:")
            for i, name in enumerate(section_names):
                bounds = sections[name]
                print(f"  [{i}] {name} (rows {bounds['data_start']}-{bounds['data_end']}, {bounds['non_empty_rows']} data rows)")
            return
    
    bounds = sections[section_name]
    data_start = bounds['data_start']
    data_end = bounds['data_end']
    
    print("="*80)
    print(f"SECTION: {section_name}")
    print("="*80)
    print(f"\nMarker row: {bounds['marker_row']}")
    print(f"Data range: rows {data_start} to {data_end}")
    print(f"Total rows: {bounds['total_rows']} (Empty: {bounds['empty_rows']}, Data: {bounds['non_empty_rows']})")
    
    # Show all data rows
    print(f"\n📊 ALL SECTION DATA (rows {data_start} to {data_end}, {bounds['total_rows']} total rows):")
    print("-"*80)
    section_data_all = df.iloc[data_start:data_end+1]
    display(section_data_all)

# Display Packages and Kits section (the first data section)
if 'sections' in locals() and sections and 'df_civic' in locals():
    if "Packages and Kits" in sections:
        print(f"Displaying: Packages and Kits (First Data Section)\n")
        display_section("Packages and Kits", sections, df_civic)
    else:
        # Show first section
        first_section = list(sections.keys())[0]
        print(f"Displaying first section: {first_section}\n")
        display_section(first_section, sections, df_civic)
else:
    print("❌ Run Step 2.1 first to identify sections")

Displaying: Packages and Kits (First Data Section)

SECTION: Packages and Kits

Marker row: 9
Data range: rows 10 to 47
Total rows: 38 (Empty: 7, Data: 31)

📊 ALL SECTION DATA (rows 10 to 47, 38 total rows):
--------------------------------------------------------------------------------


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16
10,NaN,NaN,Description,Part Number,Dealer Net,List Price,FRT,Installed Price,Application,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Comments
11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,0.0,SPORT,SPORT \nHybrid,0.0,Sport Touring Hybrid,Type R,NaN
12,NaN,25CIV-ENK-WTPK,16” Enkei M52 Wheels & Michelin X-ICE Snow Tir...,NaN,1556.88,1980,1.2,2181.6,0,0.0,0.0,•,NaN,0.0,NaN,NaN,Priced as a set of 4 wheels.
13,NaN,480-670-6545-SP or 480-670-6545-GB,"16"" Enkei M52 Wheels - Silver or Gloss Black",480-670-6545-SP or 480-670-6545-GB,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,NaN,34417,Michelin X-ICE Snow Tires - 215/55/16,34417,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16,NaN,25CIV-STL-WTPK,16” Winter Steel Wheels & Michelin X-ICE Snow ...,NaN,1124.48,1492.48,1.2,1694.08,0,0.0,0.0,•,NaN,0.0,NaN,NaN,Priced as a set of 4 wheels.
17,NaN,42700-TBA-A12,16” Winter Steel Wheels,42700-TBA-A12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18,NaN,34417,Michelin X-ICE Snow Tires - 215/55/16,34417,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [84]:
df_civic

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16
0,NaN,Honda Internal,Honda Accessories,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,Model Name,Civic Hatch,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,Model Year,2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,S = Standard (accessory comes standard on this...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,T = Tire Size Change (the size of the accessor...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142,NaN,TUCH-STICKS,Fine Sanding Sticks,TUCH-STICKS,2.45,4,0,4,0,0.0,0.0,•,•,0.0,•,•,Quantity of 5.
143,NaN,TUCH-TIPS,Paint Pen Replacement Tips,TUCH-TIPS,13.75,23,0,23,0,0.0,0.0,•,•,0.0,•,•,Quantity of 25.
144,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
145,NaN,NaN,Publication Date:,"All published prices, trim specifications, and...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [83]:
# Display section by INDEX (0-based)
# You can now use either:
#   display_section(0, sections, df_civic)  -> First section
#   display_section(1, sections, df_civic)  -> Second section
#   display_section(3, sections, df_civic)  -> Fourth section
# OR by name:
#   display_section("Packages and Kits", sections, df_civic)

if 'sections' in locals() and sections and 'df_civic' in locals():
    print("EXAMPLE: Display first 3 sections by index\n")
    
    # Show first section (index 0)
    print("\n" + "="*80)
    print("DISPLAYING SECTION BY INDEX: 0 (First section)")
    print("="*80)
    display_section(0, sections, df_civic)
    
#     # Show second section (index 1)
#     print("\n\n" + "="*80)
#     print("DISPLAYING SECTION BY INDEX: 1 (Second section)")
#     print("="*80)
#     display_section(1, sections, df_civic)
    
#     # Show third section (index 2)
#     print("\n\n" + "="*80)
#     print("DISPLAYING SECTION BY INDEX: 2 (Third section)")
#     print("="*80)
#     display_section(2, sections, df_civic)
# else:
#     print("❌ sections or df_civic not loaded. Run Step 2.1 first.")

EXAMPLE: Display first 3 sections by index


DISPLAYING SECTION BY INDEX: 0 (First section)
SECTION: Packages and Kits

Marker row: 9
Data range: rows 10 to 47
Total rows: 38 (Empty: 7, Data: 31)

📊 ALL SECTION DATA (rows 10 to 47, 38 total rows):
--------------------------------------------------------------------------------


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16
10,NaN,NaN,Description,Part Number,Dealer Net,List Price,FRT,Installed Price,Application,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Comments
11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,0.0,SPORT,SPORT \nHybrid,0.0,Sport Touring Hybrid,Type R,NaN
12,NaN,25CIV-ENK-WTPK,16” Enkei M52 Wheels & Michelin X-ICE Snow Tir...,NaN,1556.88,1980,1.2,2181.6,0,0.0,0.0,•,NaN,0.0,NaN,NaN,Priced as a set of 4 wheels.
13,NaN,480-670-6545-SP or 480-670-6545-GB,"16"" Enkei M52 Wheels - Silver or Gloss Black",480-670-6545-SP or 480-670-6545-GB,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,NaN,34417,Michelin X-ICE Snow Tires - 215/55/16,34417,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16,NaN,25CIV-STL-WTPK,16” Winter Steel Wheels & Michelin X-ICE Snow ...,NaN,1124.48,1492.48,1.2,1694.08,0,0.0,0.0,•,NaN,0.0,NaN,NaN,Priced as a set of 4 wheels.
17,NaN,42700-TBA-A12,16” Winter Steel Wheels,42700-TBA-A12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18,NaN,34417,Michelin X-ICE Snow Tires - 215/55/16,34417,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Step 2.2.1: Display Section by Index (NEW - Index Support)

## UPDATE: Additional Requirements for Step 2

### Applicability Markers (Clarification)
**All of these indicate APPLICABLE:**
- `•` (bullet) = applicable
- `E` = exception (applicable but with special conditions)
- `T` = test (applicable but marked for testing)

**These indicate NOT APPLICABLE:**
- `N/A` = explicitly not applicable
- Empty/NaN = no data = not applicable

**Filter Rule:** Keep rows where applicability is `•`, `E`, or `T`. Exclude `N/A` and empty.

### Error Flags for Orphaned Rows
Add a `error_flags` column to flag data quality issues:
```
MISSING_PART_NUMBER       - Part Number is null/empty
MISSING_DESCRIPTION       - Description is null/empty
MISSING_COST              - Cost is null/empty
MISSING_PRICE             - Price is null/empty
MISSING_PRICE_AND_COST    - Both price and cost missing
NO_APPLICABLE_TRIMS       - Part has no trims marked as applicable
INVALID_COST_FORMAT       - Cost value cannot be parsed
INVALID_PRICE_FORMAT      - Price value cannot be parsed
ORPHANED_ROW              - Row has critical missing data
```

**Logic:** 
- For each row, check for missing critical fields
- If issues found, add flags to the `error_flags` column
- Still include the row (don't drop) - for later review/fixing

### Section Column
Add a `section` column to track origin:
```
"Packages and Kits"
"Electronics"
"Interior"
"Exterior"
"Cargo and Storage"
"General Accessories"
```

This enables:
- Grouping/filtering by section
- Section-specific processing rules
- Quality reports by section
- Audit trails

## Updated Output Columns

| Column | Type | Example | Notes |
|--------|------|---------|-------|
| `Description` | string | "16\" Wheel" | Product description |
| `Part Number` | string | "25CIV-ENK-123" | Part identifier |
| `Cost` | string | "$1,234.56" | Cost value (keep as string initially) |
| `Residue` | string | (optional) | Residue value |
| `FRT` | float | 1.2 | Freight cost multiplier |
| `Price` | string | "$2,000.00" | Retail price (keep as string initially) |
| `Comments` | string | "Priced as a set..." | Notes |
| `section` | string | "Packages and Kits" | Source section |
| `trim_level` | string | "LX", "SPORT", "Si" | Which trim |
| `applicability` | string | "•", "E", "T" | Applicable? |
| `error_flags` | string | "MISSING_PRICE,INVALID_COST_FORMAT" | Quality issues (pipe-separated) |

## Updated Process Flow

```
1. Load data sheet (civic 5d EN)
   ↓
2. Extract metadata (section marker, publication date) ✓ DONE
   ↓
3. Identify section boundaries
   ↓
4. For EACH SECTION:
   a. Identify product vs trim columns
   b. Clean trim column names
   c. Extract applicability markers
   d. Detect orphaned/invalid rows → add error_flags
   e. Add section name to each row
   f. Melt to long format
   g. Filter to applicable rows (•, E, T)
   ↓
5. Combine all sections
   ↓
6. Validate column mapping
   ↓
7. Output: Flat CSV with all parts × trims
   - Includes: description, part#, cost, residue, frt, price, comments
   - Plus: section, trim_level, applicability, error_flags
```

## Implementation Ready ✓
All components defined. Ready for Step 2.1 implementation.

In [12]:
import pandas as pd
import os
from pathlib import Path
import sys
import json
import re
from datetime import datetime

# Add project to path
project_root = Path("../../../")
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root.resolve()}")
print(f"Libraries loaded successfully ✓")

Project root: C:\Users\paxm\OneDrive - PBS SYSTEMS\Desktop\Office\Projects\OEM Accessory project\OEM_Accessories_v1
Libraries loaded successfully ✓


## Step 1: Locate Honda Files

In [13]:
# Find all Honda files
landing_zone = Path("../../../landing_zone/2026/Honda")
print(f"Landing zone: {landing_zone.resolve()}\n")

# Find Excel files
excel_files = list(landing_zone.rglob("*.xlsx"))
print(f"Total Honda files found: {len(excel_files)}\n")

# Display files
for i, file in enumerate(excel_files[:10], 1):
    size_mb = file.stat().st_size / (1024**2)
    print(f"{i:2d}. {file.name:<50} ({size_mb:.2f} MB)")

if len(excel_files) > 10:
    print(f"\n... and {len(excel_files) - 10} more files")

# Use first file for testing
test_file = excel_files[0] if excel_files else None
print(f"\n✓ Using: {test_file.name if test_file else 'No file found'}")

Landing zone: C:\Users\paxm\OneDrive - PBS SYSTEMS\Desktop\Office\Projects\OEM Accessory project\OEM_Accessories_v1\landing_zone\2026\Honda

Total Honda files found: 14

 1. 2025 Honda Accessories - June25.xlsx               (0.25 MB)
 2. 2025-Honda-Master-June25.xlsx                      (0.12 MB)
 3. 2026-Honda_Master-06_2025.xlsx                     (0.04 MB)
 4. 2026_Honda_Accessories-06-2025 - Copy.xlsx         (0.11 MB)
 5. 2026_Honda_Accessories-06-2025.xlsx                (0.11 MB)
 6. Honda_ACCY_Master_2026_FR.xlsx                     (0.05 MB)
 7. accord.xlsx                                        (1.40 MB)
 8. civic 4d.xlsx                                      (3.38 MB)
 9. civic 5d.xlsx                                      (5.14 MB)
10. crv.xlsx                                           (2.21 MB)

... and 4 more files

✓ Using: 2025 Honda Accessories - June25.xlsx


## Step 2: Inspect File Sheet Structure

In [9]:
# Load Excel file info
xl = pd.ExcelFile(test_file)
sheet_names = xl.sheet_names

print(f"File: {test_file.name}")
print(f"Total sheets: {len(sheet_names)}\n")

# Categorize sheets
template_sheets = [s for s in sheet_names if s.lower() == "template"]
loadfile_sheets = [s for s in sheet_names if "loadfile" in s.lower()]
master_sheets = [s for s in sheet_names if " - master" in s.lower() or "- Master" in s]
trim_sheets = [s for s in sheet_names if s not in template_sheets + loadfile_sheets + master_sheets]

print("SHEET BREAKDOWN:")
print(f"\nTemplate sheets ({len(template_sheets)}):")
for sheet in template_sheets:
    print(f"  • {sheet}")

print(f"\nLoadFile sheets ({len(loadfile_sheets)}):")
for sheet in loadfile_sheets:
    print(f"  • {sheet}")

print(f"\nMaster sheets ({len(master_sheets)}):")
for sheet in master_sheets:
    print(f"  • {sheet}")

print(f"\nTrim/Model sheets ({len(trim_sheets)}):")
for sheet in trim_sheets[:5]:
    print(f"  • {sheet}")
if len(trim_sheets) > 5:
    print(f"  ... and {len(trim_sheets) - 5} more")

File: 2025 Honda Accessories - June25.xlsx
Total sheets: 47

SHEET BREAKDOWN:

Template sheets (1):
  • Template

LoadFile sheets (2):
  • LoadFile - Civic Sedan
  • LoadFile - Civic Hat

Master sheets (6):
  • Civic Sedan - Master
  • Pilot - Master
  • ODYSSEY - Master
  • Ridgeline - Master
  • CRV - Master
  • Accord - Master

Trim/Model sheets (38):
  • Si Civic Sedan
  • Sport Civic Sedan
  • LX Civic Sedan
  • SportHybrid - Civic Sedan
  • SportTouring - Civic Sedan
  ... and 33 more


## Step 3: Load Primary Data Sheet (Master)

In [10]:
# Select primary sheet (Master has priority)
primary_sheet = master_sheets[0] if master_sheets else loadfile_sheets[0] if loadfile_sheets else trim_sheets[0]

print(f"Selected primary sheet: {primary_sheet}\n")

# Load the data
df = pd.read_excel(test_file, sheet_name=primary_sheet)

print(f"LOADED DATA SUMMARY:")
print(f"  Rows: {len(df)}")
print(f"  Columns: {len(df.columns)}")
print(f"  Shape: {df.shape}\n")

print(f"COLUMN NAMES:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

Selected primary sheet: Civic Sedan - Master

LOADED DATA SUMMARY:
  Rows: 80
  Columns: 11
  Shape: (80, 11)

COLUMN NAMES:
   1. Description
   2. Part Number
   3. Cost
   4. FRT
   5. Price
   6. LX
   7. SPORT
   8. SPORT
Hybrid
   9. Sport
Touring
  10. Si
  11. Comments


## Step 4: Display First Few Rows (DATA INSPECTION)

In [11]:
print("FIRST 5 ROWS OF DATA:\n")
display(df.head())

print("\n\nDATA TYPES:\n")
print(df.dtypes)

FIRST 5 ROWS OF DATA:



,Description,Part Number,Cost,FRT,Price,LX,SPORT,SPORT\nHybrid,Sport\nTouring,Si,Comments
0,16” Enkei M52 Wheels & Michelin X-ICE Snow Tir,25CIV-ENK-WTPK,"$ 1,980.00",1.2,"$2,174.40",•,•,NaN,NaN,•,Priced as a set of 4 wheels. \n\nIncludes:\n16...
1,16” Winter Steel Wheels & Michelin X-ICE Snow ...,NaN,"$ 1,492.48",1.2,"$1,686.88",•,•,NaN,NaN,•,Priced as a set of 4 wheels. \n\nIncludes:\n16...
2,Aero Kit,NaN,"$ 1,289.70",3.0,"$1,775.70",•,•,•,•,NaN,10% Discount Applied. Includes body coloured f...
3,Black Emblem Set - SPORT,25CIV-MBM-SPT,$216,1.4,$442.80,NaN,•,NaN,NaN,NaN,"10% Discount Applied. \n\nIncludes:\nEmblem, G..."
4,Black Emblem Set - Sport Hybrid,25CIV-MBM-SPHY,$281.7,1.6,$540.90,NaN,NaN,•,•,NaN,"10% Discount Applied.\n\nIncludes:\nEmblem, Gl..."




DATA TYPES:

Description           str
Part Number           str
Cost                  str
FRT               float64
Price                 str
LX                    str
SPORT                 str
SPORT\nHybrid         str
Sport\nTouring        str
Si                    str
Comments              str
dtype: object


## Step 5: Identify Wide Format (Trim Columns)

In [15]:
# Identify trim columns (these are the column headers that represent trim levels)
trim_indicators = ["LX", "SPORT", "Si", "Type R", "Touring", "EX-L", "Sport", "Hybrid", "Sport L"]
trim_columns = [col for col in df.columns if any(trim in col for trim in trim_indicators)]

print(f"WIDE FORMAT DETECTION:\n")
print(f"Wide format detected: {len(trim_columns) > 0}")
print(f"\nTrim columns found ({len(trim_columns)}):")
for i, col in enumerate(trim_columns, 1):
    print(f"  {i}. {col}")

print(f"\nProduct columns (non-trim):")
product_columns = [col for col in df.columns if col not in trim_columns]
for i, col in enumerate(product_columns, 1):
    print(f"  {i}. {col}")

WIDE FORMAT DETECTION:

Wide format detected: True

Trim columns found (5):
  1. LX
  2. SPORT
  3. SPORT
Hybrid
  4. Sport
Touring
  5. Si

Product columns (non-trim):
  1. Description
  2. Part Number
  3. Cost
  4. FRT
  5. Price
  6. Comments


## Step 6: Show Wide Format Data (The Transformation Challenge)

In [16]:
# Show the wide format more clearly
print("WIDE FORMAT VISUALIZATION (First 8 rows, all columns):\n")

# Show with limited columns for visibility
display_df = df.head(8).copy()
display(display_df)

print("\n✓ Notice how trim information is in COLUMNS (LX, SPORT, SPORT Hybrid, Si, etc.)")
print("✓ Each cell contains: • (yes), NaN (no), or empty (no)")
print("✓ This needs to be MELTED to long format for Step 2")

WIDE FORMAT VISUALIZATION (First 8 rows, all columns):



,Description,Part Number,Cost,FRT,Price,LX,SPORT,SPORT\nHybrid,Sport\nTouring,Si,Comments
0,16” Enkei M52 Wheels & Michelin X-ICE Snow Tir,25CIV-ENK-WTPK,"$ 1,980.00",1.2,"$2,174.40",•,•,NaN,NaN,•,Priced as a set of 4 wheels. \n\nIncludes:\n16...
1,16” Winter Steel Wheels & Michelin X-ICE Snow ...,NaN,"$ 1,492.48",1.2,"$1,686.88",•,•,NaN,NaN,•,Priced as a set of 4 wheels. \n\nIncludes:\n16...
2,Aero Kit,NaN,"$ 1,289.70",3.0,"$1,775.70",•,•,•,•,NaN,10% Discount Applied. Includes body coloured f...
3,Black Emblem Set - SPORT,25CIV-MBM-SPT,$216,1.4,$442.80,NaN,•,NaN,NaN,NaN,"10% Discount Applied. \n\nIncludes:\nEmblem, G..."
4,Black Emblem Set - Sport Hybrid,25CIV-MBM-SPHY,$281.7,1.6,$540.90,NaN,NaN,•,•,NaN,"10% Discount Applied.\n\nIncludes:\nEmblem, Gl..."
5,Cold Weather Package - 1.5L,22CIV-COLD-PK15,$350.55,1.4,$577.35,NaN,NaN,NaN,NaN,•,5% Discount Applied. \n\nIncludes:\nEngine Blo...
6,Cold Weather Package - 2.0L,22CIV-COLD-PK20,$374.3,1.4,$601.10,•,•,•,•,NaN,5% Discount Applied.\n\nIncludes:\nEngine Bloc...
7,Protection Package - All-Season,22CIV-PRO-ALL,$536,0.6,$633.20,•,•,NaN,NaN,•,Includes:\nFloor Mats - All-Season\nTrunk Tray...



✓ Notice how trim information is in COLUMNS (LX, SPORT, SPORT Hybrid, Si, etc.)
✓ Each cell contains: • (yes), NaN (no), or empty (no)
✓ This needs to be MELTED to long format for Step 2


## Step 7: Extract Metadata from File

In [17]:
# Extract metadata
filename = test_file.name

# Extract year from filename
year_match = re.search(r'(202[5-9]|203[0-9])', filename)
model_year = year_match.group(1) if year_match else None

# Extract models from master sheet names
def extract_model_from_sheet_name(sheet_name):
    model = sheet_name.replace(" - Master", "").replace("LoadFile - ", "")
    trim_prefixes = ["Si ", "Sport ", "Sport Hybrid ", "Sport Touring ", "Type R ",
                    "Ex-L ", "EX-L ", "Lx ", "LX ", "Touring ", "Hybrid ",
                    "Black Edition ", "TrailSport ", "Sport L "]
    for prefix in trim_prefixes:
        if model.startswith(prefix):
            model = model[len(prefix):]
    return model.strip() if model.strip() else None

models_found = sorted(list(set([extract_model_from_sheet_name(s) for s in master_sheets if extract_model_from_sheet_name(s)])))

print("EXTRACTED METADATA:\n")
print(f"File: {filename}")
print(f"Model Year: {model_year}")
print(f"Models in file: {len(models_found)}")
for model in models_found:
    print(f"  • {model}")
print(f"\nPrimary sheet: {primary_sheet}")
print(f"Data shape: {len(df)} rows × {len(df.columns)} columns")
print(f"Wide format: Yes")
print(f"Trim columns: {len(trim_columns)}")

EXTRACTED METADATA:

File: 2025 Honda Accessories - June25.xlsx
Model Year: 2025
Models in file: 6
  • Accord
  • CRV
  • Civic Sedan
  • ODYSSEY
  • Pilot
  • Ridgeline

Primary sheet: Civic Sedan - Master
Data shape: 80 rows × 11 columns
Wide format: Yes
Trim columns: 5


## Step 8: Data Quality Check

In [18]:
print("DATA QUALITY ANALYSIS:\n")

# Check for null values
print("Null values per column:")
null_counts = df.isnull().sum()
for col, count in null_counts.items():
    pct = (count / len(df)) * 100
    print(f"  {col:<30} {count:3d} ({pct:5.1f}%)")

print(f"\n\nData completeness:")
for col in product_columns:
    non_null = df[col].notna().sum()
    print(f"  {col:<30} {non_null:3d}/{len(df)} rows")

print(f"\n✓ Data quality check complete")

DATA QUALITY ANALYSIS:

Null values per column:
  Description                      0 (  0.0%)
  Part Number                      2 (  2.5%)
  Cost                             0 (  0.0%)
  FRT                              0 (  0.0%)
  Price                            0 (  0.0%)
  LX                              26 ( 32.5%)
  SPORT                           19 ( 23.8%)
  SPORT
Hybrid                    19 ( 23.8%)
  Sport
Touring                   15 ( 18.8%)
  Si                              28 ( 35.0%)
  Comments                        47 ( 58.8%)


Data completeness:
  Description                     80/80 rows
  Part Number                     78/80 rows
  Cost                            80/80 rows
  FRT                             80/80 rows
  Price                           80/80 rows
  Comments                        33/80 rows

✓ Data quality check complete


## Step 9: Preview of Step 2 (Melt Operation)

In [19]:
print("PREVIEW: What Step 2 Will Do (Melt to Long Format)\n")

# Perform melt operation to show what Step 2 will do
df_melted = pd.melt(
    df,
    id_vars=product_columns,
    value_vars=trim_columns,
    var_name="trim_level",
    value_name="applicability"
)

print(f"After melting:")
print(f"  Rows: {len(df)} → {len(df_melted)} (one per trim combination)")
print(f"  Columns: {len(df.columns)} → {len(df_melted.columns)}\n")

print(f"New columns: {list(df_melted.columns)}\n")

# Filter to rows with applicability (non-null)
df_applicable = df_melted.dropna(subset=["applicability"])
print(f"After filtering applicability (null/empty removed):")
print(f"  Rows: {len(df_melted)} → {len(df_applicable)} (only applicable trims)\n")

print("MELTED DATA SAMPLE (First 15 rows):\n")
display(df_applicable.head(15))

print(f"\n✓ This is the long-format data that Step 2 will produce")

PREVIEW: What Step 2 Will Do (Melt to Long Format)

After melting:
  Rows: 80 → 400 (one per trim combination)
  Columns: 11 → 8

New columns: ['Description', 'Part Number', 'Cost', 'FRT', 'Price', 'Comments', 'trim_level', 'applicability']

After filtering applicability (null/empty removed):
  Rows: 400 → 293 (only applicable trims)

MELTED DATA SAMPLE (First 15 rows):



,Description,Part Number,Cost,FRT,Price,Comments,trim_level,applicability
0,16” Enkei M52 Wheels & Michelin X-ICE Snow Tir,25CIV-ENK-WTPK,"$ 1,980.00",1.2,"$2,174.40",Priced as a set of 4 wheels. \n\nIncludes:\n16...,LX,•
1,16” Winter Steel Wheels & Michelin X-ICE Snow ...,NaN,"$ 1,492.48",1.2,"$1,686.88",Priced as a set of 4 wheels. \n\nIncludes:\n16...,LX,•
2,Aero Kit,NaN,"$ 1,289.70",3.0,"$1,775.70",10% Discount Applied. Includes body coloured f...,LX,•
6,Cold Weather Package - 2.0L,22CIV-COLD-PK20,$374.3,1.4,$601.10,5% Discount Applied.\n\nIncludes:\nEngine Bloc...,LX,•
7,Protection Package - All-Season,22CIV-PRO-ALL,$536,0.6,$633.20,Includes:\nFloor Mats - All-Season\nTrunk Tray...,LX,•
9,Protection Package - Film,25CIV-PRO-FLM,$346.5,1.2,$540.90,10% Discount Applied. Cannot be installed with...,LX,•
10,"16"" Enkei M52 Wheel - Gloss Black",480-670-6545GB,"$ 1,000.00",1.6,"$1,259.20",Priced as a set of 4 wheels. Tires sold separa...,LX,E
11,"16"" Enkei M52 Wheel - Silver",480-670-6545SP,"$ 1,000.00",1.6,"$1,259.20",Priced as a set of 4 wheels. Tires sold separa...,LX,E
15,Wheel Locks - Black,08W42-TGG-301,$98,0.2,$130.40,NaN,LX,•
16,Wheel Locks - Chrome,08W42-SNA-301,$70,0.2,$102.40,NaN,LX,•



✓ This is the long-format data that Step 2 will produce


## Step 10: Run Official Step 1 Implementation

In [20]:
# Import the official Step 1 implementation
from accy_v2.oems.honda.pipeline.step1_validation import validate_and_extract

print("RUNNING OFFICIAL STEP 1 IMPLEMENTATION\n")

# Run it
data_dict, metadata, dq_warnings = validate_and_extract(str(test_file))

print(f"Status: {metadata.get('status')}")
print(f"\nMetadata extracted:")
print(json.dumps({k: v for k, v in metadata.items() if k not in ['file_path']}, indent=2, default=str))

if dq_warnings:
    print(f"\nDQ Warnings ({len(dq_warnings)}):")
    for warning in dq_warnings:
        print(f"  [{warning['severity']}] {warning['message']}")
else:
    print(f"\n✓ No DQ warnings")

if data_dict:
    df_loaded = data_dict['EN']
    print(f"\n✓ Data loaded: {len(df_loaded)} rows × {len(df_loaded.columns)} columns")

RUNNING OFFICIAL STEP 1 IMPLEMENTATION

Status: SUCCESS

Metadata extracted:
{
  "file_name": "2025 Honda Accessories - June25.xlsx",
  "model_year": "2025",
  "publication_date": null,
  "found_publication_date_string": false,
  "manufacture_validated": true,
  "models_in_file": [
    "Accord",
    "CRV",
    "Civic Sedan",
    "ODYSSEY",
    "Pilot",
    "Ridgeline"
  ],
  "primary_sheet": "Civic Sedan - Master",
  "sheet_summary": {
    "template_sheets": 1,
    "loadfile_sheets": 2,
    "master_sheets": 6,
    "trim_sheets": 38,
    "total_sheets": 47
  },
  "loaded_sheet": "Civic Sedan - Master",
  "data_shape": {
    "rows": 80,
    "columns": 11,
    "column_names": [
      "Description",
      "Part Number",
      "Cost",
      "FRT",
      "Price",
      "LX",
      "SPORT",
      "SPORT\nHybrid",
      "Sport\nTouring",
      "Si",
      "Comments"
    ]
  },
  "wide_format_detected": true,
  "trim_columns": [
    "LX",
    "SPORT",
    "SPORT\nHybrid",
    "Sport\nTouring",


## Summary

In [21]:
print("="*80)
print("STEP 1: LOADING SUMMARY")
print("="*80)

print(f"""
✓ File loaded successfully
  File: {test_file.name}
  Sheet: {primary_sheet}
  Data: {len(df)} rows × {len(df.columns)} columns

✓ Metadata extracted
  Model Year: {model_year}
  Models: {', '.join(models_found[:3])}{'...' if len(models_found) > 3 else ''}
  File organization: {len(template_sheets)} template, {len(loadfile_sheets)} loadfile, {len(master_sheets)} master, {len(trim_sheets)} trim sheets

✓ Format detected
  Format type: WIDE (trims as columns)
  Trim columns: {len(trim_columns)} ({', '.join(trim_columns[:3])}{'...' if len(trim_columns) > 3 else ''})
  Product columns: {len(product_columns)}

✓ Ready for Step 2
  Next: Melt wide format to long format
  Expected output: ~{len(df_applicable)} rows (one per accessory × trim combination)
""")

print("="*80)

STEP 1: LOADING SUMMARY

✓ File loaded successfully
  File: 2025 Honda Accessories - June25.xlsx
  Sheet: Civic Sedan - Master
  Data: 80 rows × 11 columns

✓ Metadata extracted
  Model Year: 2025
  Models: Accord, CRV, Civic Sedan...
  File organization: 1 template, 2 loadfile, 6 master, 38 trim sheets

✓ Format detected
  Format type: WIDE (trims as columns)
  Trim columns: 5 (LX, SPORT, SPORT
Hybrid...)
  Product columns: 6

✓ Ready for Step 2
  Next: Melt wide format to long format
  Expected output: ~293 rows (one per accessory × trim combination)



In [40]:
# Step 11: Test metadata extraction on the currently loaded df (from Step 3)

# Use the reloaded module
extract_metadata_from_data_sheet = step1_module.extract_metadata_from_data_sheet

print("="*80)
print("TESTING METADATA EXTRACTION")
print("="*80)

metadata = extract_metadata_from_data_sheet(df)

print(f"\nResults:")
print(f"  Honda Accessories Found: {metadata.get('honda_accessories_found')}")
print(f"  Section Marker Row: {metadata.get('section_marker_row')}")
print(f"  Publication Date: {metadata.get('publication_date')}")

if metadata.get('section_marker_row') is not None:
    section_row = metadata['section_marker_row']
    print(f"\n✓ Section marker found at row {section_row}")
else:
    print(f"\n⚠️  No section marker found in this file")

TESTING METADATA EXTRACTION

Results:
  Honda Accessories Found: False
  Section Marker Row: None
  Publication Date: None

⚠️  No section marker found in this file


In [41]:
# Step 11: Test metadata extraction on the currently loaded df (from Step 3)

try:
    from accy_v2.oems.honda.pipeline.step1_validation import extract_metadata_from_data_sheet
    print("✓ Import successful")
except ImportError as e:
    print(f"❌ Import error: {e}")
    import sys
    print(f"Python path: {sys.path[0:3]}")

# Now test with df (should be loaded from Step 3)
print("\n" + "="*80)
print("TESTING METADATA EXTRACTION")
print("="*80)

metadata = extract_metadata_from_data_sheet(df)

print(f"\nResults:")
print(f"  Honda Accessories Found: {metadata.get('honda_accessories_found')}")
print(f"  Section Marker Row: {metadata.get('section_marker_row')}")
print(f"  Publication Date: {metadata.get('publication_date')}")

if metadata.get('section_marker_row') is not None:
    section_row = metadata['section_marker_row']
    print(f"\n✓ Section marker found at row {section_row}")

✓ Import successful

TESTING METADATA EXTRACTION

Results:
  Honda Accessories Found: False
  Section Marker Row: None
  Publication Date: None


In [37]:
# Debug: Show first and last rows to understand file structure

print("="*80)
print(f"FILE: {test_file.name}")
print(f"SHEET: {primary_sheet}")
print("="*80)

print("\n📍 FIRST 3 ROWS (HEADER):")
print("-"*80)
display(df.head(3))

print("\n📍 LAST 3 ROWS (FOOTER):")
print("-"*80)
display(df.tail(3))

print("\n📍 LOOKING FOR SECTION MARKERS:")
df_str = df.astype(str)
markers_found = []
for idx, row in df_str.iterrows():
    row_text = " ".join(str(val) for val in row.values)
    if "Packages" in row_text or "Groupes" in row_text or "1.0" in row_text:
        markers_found.append((idx, row_text[:100]))

if markers_found:
    print(f"Found {len(markers_found)} rows with 'Packages', 'Groupes', or '1.0':")
    for idx, text in markers_found[:3]:
        print(f"  Row {idx}: {text}")
else:
    print("❌ No section markers found in this file")

FILE: 2025 Honda Accessories - June25.xlsx
SHEET: Civic Sedan - Master

📍 FIRST 3 ROWS (HEADER):
--------------------------------------------------------------------------------


,Description,Part Number,Cost,FRT,Price,LX,SPORT,SPORT\nHybrid,Sport\nTouring,Si,Comments
0,16” Enkei M52 Wheels & Michelin X-ICE Snow Tir,25CIV-ENK-WTPK,"$ 1,980.00",1.2,"$2,174.40",•,•,NaN,NaN,•,Priced as a set of 4 wheels. \n\nIncludes:\n16...
1,16” Winter Steel Wheels & Michelin X-ICE Snow ...,NaN,"$ 1,492.48",1.2,"$1,686.88",•,•,NaN,NaN,•,Priced as a set of 4 wheels. \n\nIncludes:\n16...
2,Aero Kit,NaN,"$ 1,289.70",3.0,"$1,775.70",•,•,•,•,NaN,10% Discount Applied. Includes body coloured f...



📍 LAST 3 ROWS (FOOTER):
--------------------------------------------------------------------------------


,Description,Part Number,Cost,FRT,Price,LX,SPORT,SPORT\nHybrid,Sport\nTouring,Si,Comments
77,Fine Sanding Needles,TUCH-NEEDLES,$15.75,0.0,$15.75,•,•,•,•,•,Quantity of 5.
78,Fine Sanding Sticks,TUCH-STICKS,$4,0.0,$4,•,•,•,•,•,Quantity of 5.
79,Paint Pen Replacement Tips,TUCH-TIPS,$23,0.0,$23,•,•,•,•,•,Quantity of 25.



📍 LOOKING FOR SECTION MARKERS:
Found 9 rows with 'Packages', 'Groupes', or '1.0':
  Row 32: Underbody Spoiler, HPD - Rear 08F24-T20-100 $373 1.0 $535.00 • • • • • HPD Logo.
  Row 33: Underbody Spoiler, HPD - Side 08F04-T20-100 $635 1.0 $797.00 • • • • nan HPD Logo.
  Row 34: Platinum White Pearl (NH-883P) 08F01-T20-130A $425 1.0 $587 • • • • • nan


## Debug: Check What's Actually in the Data

## Step 11: Test Metadata Extraction Helper

In [42]:
# STEP 5: Display last 3 rows (footer/closing data)

if civic_5d_file and en_sheets_civic5d:
    print("\n" + "="*80)
    print("FOOTER DATA (Last 3 Rows)")
    print("="*80)
    
    for en_sheet in en_sheets_civic5d:
        print(f"\n{'='*80}")
        print(f"EN Sheet: {en_sheet}")
        print(f"{'='*80}\n")
        
        df_en_full = pd.read_excel(civic_5d_file, sheet_name=en_sheet)
        print(f"Last 3 rows (out of {len(df_en_full)} total):")
        print("-"*80)
        display(df_en_full.tail(3))

if civic_5d_file and fr_sheets_civic5d:
    print(f"\n\n{'='*80}")
    print(f"FR Sheet: {fr_sheets_civic5d[0]}")
    print(f"{'='*80}\n")
    
    df_fr_full = pd.read_excel(civic_5d_file, sheet_name=fr_sheets_civic5d[0])
    print(f"Last 3 rows (out of {len(df_fr_full)} total):")
    print("-"*80)
    display(df_fr_full.tail(3))


FOOTER DATA (Last 3 Rows)

EN Sheet: 26MY_CIV5D_APP_EN

Last 3 rows (out of 147 total):
--------------------------------------------------------------------------------


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16
144,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
145,NaN,NaN,Publication Date:,"All published prices, trim specifications, and...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
146,NaN,NaN,2025-07-21 00:00:00,Red Pricing = Promotional Pricing - offered fo...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN




FR Sheet: 26MY_CIV5D_APP_FR

Last 3 rows (out of 148 total):
--------------------------------------------------------------------------------


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16
145,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
146,NaN,NaN,Date de publication:,"Tout prix, spécifications de versions et dispo...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
147,NaN,NaN,2025-07-21 00:00:00,*Prix en rouge = Prix promotionnels offerts po...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
# STEP 4: Extract metadata rows (before section markers) and footer rows

if civic_5d_file and en_sheets_civic5d:
    print("\n" + "="*80)
    print("METADATA EXTRACTION (Rows BEFORE Section Markers)")
    print("="*80)
    
    for en_sheet in en_sheets_civic5d:
        print(f"\n{'='*80}")
        print(f"EN Sheet: {en_sheet}")
        print(f"{'='*80}\n")
        
        df_en_full = pd.read_excel(civic_5d_file, sheet_name=en_sheet)
        df_en_str = df_en_full.astype(str)
        
        # Find row with "Packages and Kits"
        packages_row_idx = None
        for idx, row in df_en_str.iterrows():
            row_text = " ".join(str(val) for val in row.values)
            if "Packages and Kits" in row_text or "1.0 Packages" in row_text:
                packages_row_idx = idx
                break
        
        if packages_row_idx is not None:
            print(f"Found 'Packages and Kits' at row {packages_row_idx}\n")
            print("Rows BEFORE this section (METADATA):")
            print("-"*80)
            metadata_df = df_en_full.iloc[0:packages_row_idx]
            display(metadata_df)
        else:
            print("'Packages and Kits' marker not found")
            print("Showing first 10 rows:")
            display(df_en_full.head(10))

if civic_5d_file and fr_sheets_civic5d:
    print(f"\n\n{'='*80}")
    print(f"FR Sheet: {fr_sheets_civic5d[0]}")
    print(f"{'='*80}\n")
    
    df_fr_full = pd.read_excel(civic_5d_file, sheet_name=fr_sheets_civic5d[0])
    df_fr_str = df_fr_full.astype(str)
    
    # Find row with "Groupes et ensembles" (FR for "Packages and Kits")
    groupes_row_idx = None
    for idx, row in df_fr_str.iterrows():
        row_text = " ".join(str(val) for val in row.values)
        if "Groupes et ensembles" in row_text or "1.0 Groupes" in row_text:
            groupes_row_idx = idx
            break
    
    if groupes_row_idx is not None:
        print(f"Found 'Groupes et ensembles' at row {groupes_row_idx}\n")
        print("Rows BEFORE this section (METADATA):")
        print("-"*80)
        metadata_df_fr = df_fr_full.iloc[0:groupes_row_idx]
        display(metadata_df_fr)
    else:
        print("'Groupes et ensembles' marker not found")
        print("Showing first 10 rows:")
        display(df_fr_full.head(10))


METADATA EXTRACTION (Rows BEFORE Section Markers)

EN Sheet: 26MY_CIV5D_APP_EN

Found 'Packages and Kits' at row 9

Rows BEFORE this section (METADATA):
--------------------------------------------------------------------------------


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16
0,NaN,Honda Internal,Honda Accessories,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,Model Name,Civic Hatch,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,Model Year,2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,S = Standard (accessory comes standard on this...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,T = Tire Size Change (the size of the accessor...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,E = Exchange Standard Equipment (the size of t...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,N/A = Not Applicable (accessory cannot be appl...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Province,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN




FR Sheet: 26MY_CIV5D_APP_FR

Found 'Groupes et ensembles' at row 9

Rows BEFORE this section (METADATA):
--------------------------------------------------------------------------------


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16
0,NaN,Honda Internal,Accessoires Honda,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,Nom du modèle,Civic à Hayon,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,Année modèle,2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,S = De série (accessoire de série sur cette ve...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,T = Taille des pneus (la taille de la jante ac...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,E = Échange d’équipement de série (la taille d...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,N/A = Sans objet (l’accessoire ne peut être ut...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Province,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [44]:
# Load civic 5d.xlsx and extract EN sheet

# Find civic 5d file
civic_5d_files = [f for f in excel_files if "civic" in f.name.lower() and "5d" in f.name.lower()]

if not civic_5d_files:
    print("❌ civic 5d.xlsx not found in landing zone")
else:
    civic_5d_file = civic_5d_files[0]
    print(f"✓ Found: {civic_5d_file.name}\n")
    
    # List sheets
    xl = pd.ExcelFile(civic_5d_file)
    print(f"Sheets in file: {xl.sheet_names}\n")
    
    # Find EN sheet
    en_sheets = [s for s in xl.sheet_names if "_EN" in s.upper()]
    
    if en_sheets:
        en_sheet = en_sheets[0]
        print(f"Loading EN sheet: {en_sheet}\n")
        
        df_civic = pd.read_excel(civic_5d_file, sheet_name=en_sheet)
        print(f"✓ Loaded: {len(df_civic)} rows × {len(df_civic.columns)} columns")
    else:
        print("❌ No EN sheet found")
        df_civic = None

✓ Found: civic 5d.xlsx

Sheets in file: ['SELECT PROVINCE', '26MY_CIV5D_APP_EN', '26MY_CIV5D_APP_FR', '26MY_CIV5D_SELL-EN', '26MY_CIV5D_SELL-FR']

Loading EN sheet: 26MY_CIV5D_APP_EN

✓ Loaded: 147 rows × 17 columns


In [45]:
# Test metadata extraction on civic 5d EN sheet

if 'df_civic' in locals() and df_civic is not None:
    print("="*80)
    print("METADATA EXTRACTION - CIVIC 5D")
    print("="*80)
    
    metadata = extract_metadata_from_data_sheet(df_civic)
    
    print(f"\nResults:")
    print(f"  Honda Accessories Found: {metadata.get('honda_accessories_found')}")
    print(f"  Section Marker Row: {metadata.get('section_marker_row')}")
    print(f"  Publication Date: {metadata.get('publication_date')}")
    
    if metadata.get('section_marker_row') is not None:
        section_row = metadata['section_marker_row']
        print(f"\n✓ Section marker found at row {section_row}")
        print(f"  Header rows: 0 to {section_row-1} ({section_row} rows)")
        print(f"  Data rows: {section_row} to {len(df_civic)-1} ({len(df_civic)-section_row} rows)")
        
        print(f"\n📍 Header rows (first 5):")
        print("-"*80)
        display(df_civic.iloc[0:min(5, section_row)])
    else:
        print("\n❌ No section marker found")
        print("\nShowing first 5 rows:")
        display(df_civic.head(5))
else:
    print("❌ df_civic not loaded. Run the previous cell first.")

METADATA EXTRACTION - CIVIC 5D

Results:
  Honda Accessories Found: True
  Section Marker Row: 9
  Publication Date: 2025-07-21

✓ Section marker found at row 9
  Header rows: 0 to 8 (9 rows)
  Data rows: 9 to 146 (138 rows)

📍 Header rows (first 5):
--------------------------------------------------------------------------------


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16
0,NaN,Honda Internal,Honda Accessories,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,Model Name,Civic Hatch,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,Model Year,2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,S = Standard (accessory comes standard on this...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,T = Tire Size Change (the size of the accessor...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [54]:
# Comprehensive metadata validation for all sheets

def validate_sheet_metadata(df, sheet_name):
    """Extract and validate metadata from a sheet."""
    meta = {
        "sheet_name": sheet_name,
        "data_shape": {
            "rows": len(df),
            "columns": len(df.columns),
            "shape": f"{len(df)} rows × {len(df.columns)} columns"
        },
        "column_names": list(df.columns),
        "extracted_metadata": extract_metadata_from_data_sheet(df)
    }
    return meta

# Test with CIVIC 5D EN sheet
if 'df_civic' in locals() and df_civic is not None:
    print("="*80)
    print("COMPREHENSIVE METADATA VALIDATION")
    print("="*80)
    
    sheet_meta = validate_sheet_metadata(df_civic, en_sheet)
    
    print(f"\n📊 SHEET: {sheet_meta['sheet_name']}")
    print(f"   {sheet_meta['data_shape']['shape']}")
    
    print(f"\n📋 COLUMN NAMES ({len(sheet_meta['column_names'])}):")
    # for i, col in enumerate(sheet_meta['column_names'][:10], 1):
    #     print(f"   {i:2d}. {col}")
    # if len(sheet_meta['column_names']) > 10:
    #     print(f"   ... and {len(sheet_meta['column_names']) - 10} more")
    
    extracted = sheet_meta['extracted_metadata']
    print(f"\n🔍 EXTRACTED METADATA:")
    print(f"   Honda Accessories Found: {extracted.get('honda_accessories_found')}")
    print(f"   Section Marker Row: {extracted.get('section_marker_row')}")
    print(f"   Publication Date: {extracted.get('publication_date')}")
    # print(f"   Data Shape: {extracted.get('data_shape')}")
    
    # # Validation checks
    # print(f"\n✅ VALIDATION CHECKS:")
    # checks = [
    #     ("Data loaded", len(df_civic) > 0),
    #     ("Columns found", len(df_civic.columns) > 0),
    #     ("Section marker found", extracted.get('section_marker_row') is not None),
    #     ("Honda signature found", extracted.get('honda_accessories_found')),
    #     ("Publication date found", extracted.get('publication_date') is not None),
    # ]
    
    # for check_name, result in checks:
    #     status = "✓" if result else "✗"
    #     print(f"   {status} {check_name}")
    
    # all_passed = all(result for _, result in checks)
    
    # if all_passed:
    #     print(f"\n🎯 ALL CHECKS PASSED - Ready for next steps!")
    # else:
    #     print(f"\n⚠️  Some checks failed - review metadata extraction")
        
else:
    print("❌ df_civic not loaded")

COMPREHENSIVE METADATA VALIDATION

📊 SHEET: 26MY_CIV5D_APP_EN
   147 rows × 17 columns

📋 COLUMN NAMES (17):

🔍 EXTRACTED METADATA:
   Honda Accessories Found: True
   Section Marker Row: 9
   Publication Date: 2025-07-21
